In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv
/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv


In [2]:
movies_path = "/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv"
credits_path = "/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv"

movies_df = pd.read_csv(movies_path)
credits_df = pd.read_csv(credits_path)


In [3]:
movies_df.info()
credits_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [4]:
movies_df.head()
credits_df.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


## 2.1 Análisis Inicial

Se utilizó `.info()` para identificar los tipos de datos y `.head()` para observar ejemplos de los valores.

Se observó que varias columnas tienen tipo `object` pero no todas contienen JSON, por lo que se analizaron manualmente.

Columnas con valores múltiples en formato JSON:

En movies_df:
- genres
- keywords
- production_companies
- production_countries
- spoken_languages

En credits_df:
- cast
- crew

Estas columnas contienen listas de diccionarios dentro de una sola celda, lo cual representa múltiples valores en un solo atributo.

Esto viola la atomicidad requerida en bases de datos relacionales.

### Relación con 1NF

La Primera Forma Normal (1NF) establece que cada celda debe contener un valor atómico.

Las columnas identificadas no cumplen con 1NF porque contienen listas de valores (JSON) en una sola celda.

Por lo tanto, es necesario separar estos datos en tablas independientes para eliminar los grupos repetitivos.

## 2.2 Diagrama Entidad-Relación

![ER Diagram](https://i.imgur.com/1BkjzOT.png)

El diagrama E/R representa las entidades principales del sistema, como Movie, Actor, Genre, Company, Country, Language y Crew.

Se identificaron relaciones muchos-a-muchos entre Movie y las demás entidades debido a que una película puede tener múltiples actores, géneros, compañías, etc.

Estas relaciones serán resueltas en el modelo relacional mediante tablas intermedias.

## 2.3 Modelo Relacional y Normalización

Luego de identificar las columnas JSON, se tradujo el modelo E/R a un modelo relacional. La idea principal es separar los datos repetidos en tablas diferentes para que cada tabla represente una sola entidad o relación.

### Esquema lógico inicial

Movie(
    movie_id PK,
    title,
    budget,
    homepage,
    original_language,
    original_title,
    overview,
    popularity,
    release_date,
    revenue,
    runtime,
    status,
    tagline,
    vote_average,
    vote_count
)

Genre(
    genre_id PK,
    genre_name
)

Movie_Genre(
    movie_id FK,
    genre_id FK,
    PK(movie_id, genre_id)
)

Keyword(
    keyword_id PK,
    keyword_name
)

Movie_Keyword(
    movie_id FK,
    keyword_id FK,
    PK(movie_id, keyword_id)
)

Production_Company(
    company_id PK,
    company_name
)

Movie_Production_Company(
    movie_id FK,
    company_id FK,
    PK(movie_id, company_id)
)

Production_Country(
    country_code PK,
    country_name
)

Movie_Production_Country(
    movie_id FK,
    country_code FK,
    PK(movie_id, country_code)
)

Spoken_Language(
    language_code PK,
    language_name
)

Movie_Spoken_Language(
    movie_id FK,
    language_code FK,
    PK(movie_id, language_code)
)

Actor(
    actor_id PK,
    actor_name,
    gender
)

Movie_Actor(
    movie_id FK,
    actor_id FK,
    character,
    cast_order,
    PK(movie_id, actor_id, character)
)

Crew_Member(
    crew_id PK,
    crew_name,
    gender
)

Movie_Crew(
    movie_id FK,
    crew_id FK,
    department,
    job,
    PK(movie_id, crew_id, department, job)
)

### 1NF - Primera Forma Normal

Para cumplir con 1NF, cada celda debe contener un solo valor atómico. En el dataset original, varias columnas no cumplen esto porque contienen listas JSON.

Columnas con valores múltiples:
- genres
- keywords
- production_companies
- production_countries
- spoken_languages
- cast
- crew

Ejemplo:
La columna `cast` contiene varios actores dentro de una sola celda. Cada actor tiene datos como `cast_id`, `character`, `name` y `gender`.

Esto se corrigió separando esas listas en tablas nuevas como `Actor` y `Movie_Actor`.

Así se eliminan los grupos repetitivos y cada fila representa una sola relación.

### 2NF - Segunda Forma Normal

Para cumplir con 2NF, se eliminaron dependencias parciales.

En las tablas intermedias se usan llaves compuestas, por ejemplo:

Movie_Actor(movie_id, actor_id)

Si se dejara `actor_name` dentro de `Movie_Actor`, habría una dependencia parcial porque `actor_name` depende solo de `actor_id`, no de toda la llave compuesta.

Por eso se separó así:

Actor(actor_id, actor_name, gender)

Movie_Actor(movie_id, actor_id, character, cast_order)

Lo mismo se aplicó para géneros, keywords, compañías, países e idiomas. Los nombres se guardan en tablas maestras/catálogos y las relaciones con películas se guardan en tablas intermedias.

### 3NF - Tercera Forma Normal

Para cumplir con 3NF, se eliminaron dependencias transitivas. Esto significa que los atributos no clave no deben depender de otro atributo no clave.

Ejemplo:
En vez de guardar `country_name` directamente en la película, se crea una tabla separada:

Production_Country(country_code, country_name)

Luego la película se relaciona con el país usando:

Movie_Production_Country(movie_id, country_code)

De esta forma, `country_name` depende de `country_code`, no directamente de `movie_id`.

También se aplicó la misma idea con:
- genre_name depende de genre_id
- keyword_name depende de keyword_id
- company_name depende de company_id
- language_name depende de language_code
- actor_name depende de actor_id
- crew_name depende de crew_id

### BCNF / 4NF

Para BCNF y 4NF, se identificaron dependencias multivaluadas independientes.

Una película puede tener:
- muchos géneros
- muchos actores
- muchas compañías productoras
- muchos países de producción
- muchos idiomas
- muchos miembros del crew

Estas listas son independientes entre sí. Por ejemplo, los actores de una película no dependen de sus géneros. Si se dejaran todos juntos en una sola tabla, se crearían muchas repeticiones innecesarias.

Por eso se separaron en tablas intermedias:

- Movie_Genre
- Movie_Keyword
- Movie_Production_Company
- Movie_Production_Country
- Movie_Spoken_Language
- Movie_Actor
- Movie_Crew

Esto evita redundancia y hace que el diseño sea más limpio.

## 2.4 Entrega Visual

En esta sección se presentan los diagramas desarrollados para el modelado conceptual y lógico de la base de datos.

### Diagrama Entidad-Relación (E/R)

El siguiente diagrama representa el modelo conceptual, donde se identifican las entidades principales y sus relaciones (muchos a muchos) derivadas del análisis del dataset.

![ER Diagram](https://i.imgur.com/1BkjzOT.png)

---

### Modelo Relacional Final

El siguiente diagrama muestra el modelo lógico resultante luego del proceso de normalización (1NF, 2NF, 3NF y 4NF).

Se incluyen:
- Tablas principales (Movie, Actor, Genre, etc.)
- Tablas intermedias para resolver relaciones muchos-a-muchos
- Llaves primarias (PK) y foráneas (FK)

![Relational Model](https://i.imgur.com/Xp6d8ZB.png)